# 18. 새로운 데이터 분할로 최종 검증

## 이번 노트북에서 할 것
- Tox21을 새로운 random seed(예: 7)로 재분할해 완전히 독립적인
  train/valid/test 확보 (기존 seed=42 분할은 오늘 개발 과정에서 반복 참조됨)
- 확정된 라이브러리 11개 규칙으로 held-out(신규 test) 커버리지 재측정
- 단일/다중 문제 분자 재검증 (성공률, 부분개선 포함)
- 3-endpoint(Tox21/Ames/hERG) 통계 재계산
- 이 결과를 제안서 4번 섹션의 최종 수치로 확정

## 간략한 정리 (17까지)
- 라이브러리 11개 규칙 확정: nitro_group, aldehyde, Michael_acceptor_1,
  acid_halide, alkyl_halide, aniline, Sulfonic_acid_2, imine_1_oxime,
  imine_1_general, catechol, Thiocarbonyl_group
- 결합절단형(8개, rdMMPA+molzip)과 원자편집형(3개, RWMol 기반: catechol,
  Thiocarbonyl_group, imine 계열) 두 방식 병행
- 전체 데이터셋 기준 전수 검증 완료, 이름 불일치 버그 등 완전히 정리됨
- ==방법론 개선: test set을 오늘 개발 과정에서 반복 사용했음을 인지,
  새 분할로 최종 검증하기로 결정==
- 오늘까지의 정량 결과(94% 개선, 167개 통계 등)는 "탐색적 참고치"로 재분류

## 다음에 해야 할 것 (오늘 끝나면)
- 이 노트북 결과를 제안서 4번(평가) 섹션 최종 수치로 반영
- 제안서 hwpx 작업 (학생 진행 중)

In [ ]:
# 셀 1
!pip install rdkit -q
!pip install fuzzywuzzy python-Levenshtein -q
!pip install PyTDC --no-deps -q
!pip install PyYAML tqdm requests -q
!pip install openai -q

In [ ]:
# 셀 2
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')

!git clone https://{token}@github.com/Dec32th/laidd-2026.git
%cd /content/laidd-2026
!pwd

Cloning into 'laidd-2026'...
remote: Enumerating objects: 204, done.
remote: Counting objects: 100% (204/204), done.
remote: Compressing objects: 100% (145/145), done.
remote: Total 204 (delta 101), reused 143 (delta 53), pack-reused 0 (from 0)
Receiving objects: 100% (204/204), 584.68 KiB | 10.83 MiB/s, done.
Resolving deltas: 100% (101/101), done.
/content/laidd-2026
/content/laidd-2026


In [ ]:
# 셀 3
!git config --global user.email "hyekyeong.w@gmail.com"
!git config --global user.name "Dec32th"

In [ ]:
# 셀 4
import importlib
import random
import numpy as np
import pandas as pd
from collections import Counter
from scipy import stats
from rdkit import Chem
from rdkit.Chem import rdMMPA, rdFingerprintGenerator, QED
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

import src.tools.replacement_library
import src.tools.molecule_editor
import src.tools.atom_editor
import src.tools.toxicophore_detector
import src.tools.agent

from src.tools.data_prep import load_tox21_clean
from src.tools.toxicophore_detector import detect_toxicophores
from src.tools.replacement_library import get_replacement_candidates
from src.tools.molecule_editor import find_core_and_target, reassemble_molecule, propose_fix, canonicalize, iterative_fix_loop
from src.tools.agent import ask_llm_which_problem_to_fix, ask_llm_which_candidate_to_use
from src.tools.atom_editor import apply_atom_edit_from_rule

_generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
def smiles_to_ecfp(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return _generator.GetFingerprintAsNumPy(mol) if mol else None

print("도구 로드 확인 완료")

도구 로드 확인 완료


In [ ]:
# 셀 5 — 새로운 시드로 Tox21 재분할 (핵심)
data = load_tox21_clean(random_state=7)
print("새 분할 완료 (seed=7)")
print("Train:", data['X_train'].shape, "Valid:", data['X_valid'].shape, "Test:", data['X_test'].shape)

[05:03:19] WARNING: not removing hydrogen atom without neighbors
[05:03:19] Explicit valence for atom # 8 Al, 6, is greater than permitted
[05:03:19] Explicit valence for atom # 3 Al, 6, is greater than permitted
[05:03:19] Explicit valence for atom # 4 Al, 6, is greater than permitted
[05:03:19] Explicit valence for atom # 4 Al, 6, is greater than permitted
[05:03:19] Explicit valence for atom # 9 Al, 6, is greater than permitted
[05:03:19] Explicit valence for atom # 5 Al, 6, is greater than permitted
[05:03:19] Explicit valence for atom # 16 Al, 6, is greater than permitted
[05:03:19] Explicit valence for atom # 20 Al, 6, is greater than permitted


전체: 7831개, 파싱 성공: 7823개, 파싱 실패(제외): 8개


[05:03:20] WARNING: not removing hydrogen atom without neighbors


새 분할 완료 (seed=7)
Train: (5476, 2048) Valid: (1173, 2048) Test: (1174, 2048)


In [ ]:
# 셀 6 — Tox21/Ames/hERG baseline 재학습 (새 분할 기준)
X_train, y_train, w_train = data['X_train'], data['y_train'], data['w_train']
task_cols = data['task_cols']
classifiers = {}
for i, task in enumerate(task_cols):
    train_mask = w_train[:, i] == 1
    clf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
    clf.fit(X_train[train_mask], y_train[train_mask, i])
    classifiers[task] = clf

test_auc_scores = {}
for i, task in enumerate(task_cols):
    test_mask = data['w_test'][:, i] == 1
    probs = classifiers[task].predict_proba(data['X_test'][test_mask])[:, 1]
    test_auc_scores[task] = roc_auc_score(data['y_test'][test_mask, i], probs)
print(f"Tox21 baseline 완료, 새 분할 기준 평균 Test AUROC: {np.mean(list(test_auc_scores.values())):.3f}")

from tdc.single_pred import Tox

def prepare_split_generic(df):
    df = df.copy()
    df['mol_valid'] = df['Drug'].apply(lambda s: Chem.MolFromSmiles(s) is not None)
    df_clean = df[df['mol_valid']].reset_index(drop=True)
    X = np.stack(df_clean['Drug'].apply(smiles_to_ecfp).values)
    y = df_clean['Y'].values
    return X, y, df_clean['Drug'].values

ames_split = Tox(name='AMES').get_split()
X_train_ames, y_train_ames, _ = prepare_split_generic(ames_split['train'])
ames_clf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
ames_clf.fit(X_train_ames, y_train_ames)
print("Ames baseline 완료 (TDC 표준 분할)")

herg_split = Tox(name='hERG').get_split()
X_train_herg, y_train_herg, _ = prepare_split_generic(herg_split['train'])
herg_clf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
herg_clf.fit(X_train_herg, y_train_herg)
print("hERG baseline 완료 (TDC 표준 분할)")

def predict_ames(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    return ames_clf.predict_proba(smiles_to_ecfp(smiles).reshape(1, -1))[0][1]

def predict_herg(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    return herg_clf.predict_proba(smiles_to_ecfp(smiles).reshape(1, -1))[0][1]

def predict_tox21_avg(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    fp = smiles_to_ecfp(smiles).reshape(1, -1)
    return np.mean([classifiers[t].predict_proba(fp)[0][1] for t in task_cols])

Found local copy...
Loading...
Done!


Tox21 baseline 완료, 새 분할 기준 평균 Test AUROC: 0.800


Found local copy...
Loading...
Done!


Ames baseline 완료 (TDC 표준 분할)


[05:04:17] WARNING: not removing hydrogen atom without neighbors
[05:04:17] WARNING: not removing hydrogen atom without neighbors
[05:04:17] WARNING: not removing hydrogen atom without neighbors
[05:04:17] WARNING: not removing hydrogen atom without neighbors


hERG baseline 완료 (TDC 표준 분할)


In [ ]:
# 셀 7 — Qwen 연결
from openai import OpenAI
dashscope_key = userdata.get('DASHSCOPE_API_KEY')
client_qwen = OpenAI(api_key=dashscope_key, base_url="https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1")
print("Qwen 클라이언트 준비 완료")

Qwen 클라이언트 준비 완료


In [ ]:
count_known_valid = 0
for s in data['smiles_valid']:
    p = detect_toxicophores(s)
    known_count = sum(1 for x in p if get_replacement_candidates(x['rule_name']) is not None)
    if known_count >= 1:
        count_known_valid += 1

print(f"Valid set 커버리지: {count_known_valid}개 / {len(data['smiles_valid'])}개 ({count_known_valid/len(data['smiles_valid'])*100:.1f}%)")

Valid set 커버리지: 298개 / 1173개 (25.4%)


In [ ]:
single_known_valid = []
multi_known_valid = []

for s in data['smiles_valid']:
    p = detect_toxicophores(s)
    known_count = sum(1 for x in p if get_replacement_candidates(x['rule_name']) is not None)
    if known_count == 1:
        single_known_valid.append(s)
    elif known_count >= 2:
        multi_known_valid.append(s)

print(f"단일 문제 분자: {len(single_known_valid)}개")
print(f"다중 문제 분자: {len(multi_known_valid)}개")

단일 문제 분자: 271개
다중 문제 분자: 27개


In [ ]:
random.seed(42)
sample_single_valid = random.sample(single_known_valid, min(50, len(single_known_valid)))

rule_based_results_valid = []
for smi in sample_single_valid:
    result = iterative_fix_loop(smi, max_iterations=10)
    rule_based_results_valid.append({"smiles": smi, "status": result['status'], "steps": len(result['history'])-1})

status_counts_valid = Counter(r['status'] for r in rule_based_results_valid)
print("규칙기반 결과 (valid set 50개 표본):")
for status, count in status_counts_valid.items():
    print(f"  {status}: {count}개 ({count/len(rule_based_results_valid)*100:.1f}%)")

total_valid = len(rule_based_results_valid)
success_valid = sum(1 for r in rule_based_results_valid if r['status'] == 'success')
partial_valid = sum(1 for r in rule_based_results_valid if r['status'] == 'no_known_fix' and r['steps'] >= 1)
print(f"\n완전 해결: {success_valid}개 ({success_valid/total_valid*100:.1f}%)")
print(f"부분 진전: {partial_valid}개 ({partial_valid/total_valid*100:.1f}%)")
print(f"최소 1단계 이상 개선: {(success_valid+partial_valid)/total_valid*100:.1f}%")

[05:04:23] Incomplete atom labelling, cannot make bond


규칙기반 결과 (valid set 50개 표본):
  success: 24개 (48.0%)
  no_known_fix: 24개 (48.0%)
  stuck: 2개 (4.0%)

완전 해결: 24개 (48.0%)
부분 진전: 24개 (48.0%)
최소 1단계 이상 개선: 96.0%


[05:04:24] Incomplete atom labelling, cannot make bond


In [ ]:
stuck_cases_valid = [r for r in rule_based_results_valid if r['status'] == 'stuck']
print(f"stuck 케이스: {len(stuck_cases_valid)}개\n")

for r in stuck_cases_valid:
    detail = iterative_fix_loop(r['smiles'], max_iterations=10)
    last_step = detail['history'][-1]
    print(f"분자: {r['smiles'][:60]}")
    print(f"  마지막 문제들: {[p['rule_name'] for p in last_step.get('problems', [])]}")
    print(f"  실패 이유: {detail.get('reason', 'N/A')}")
    print()

stuck 케이스: 2개

분자: CCOC(=O)/C=C(\C)O[Ti](O/C(C)=C/C(=O)OCC)(OC(C)C)OC(C)C
  마지막 문제들: ['acyclic_C=C-O', 'Michael_acceptor_1']
  실패 이유: 'Michael_acceptor_1' 치환 실패

분자: CCCCN=C=O
  마지막 문제들: ['aldehyde', 'Aliphatic_long_chain']
  실패 이유: 'aldehyde' 치환 실패



In [ ]:
%%writefile src/tools/replacement_library.py

REPLACEMENT_LIBRARY = {
    "nitro_group": {
        "problem_smarts": "[N+](=O)[O-]",
        "candidates": [
            {"smiles": "N", "name": "primary amine",
             "rationale": "극성을 유지하면서 니트로기의 환원성 대사 중간체 생성 경로를 제거함"},
            {"smiles": "S(=O)(=O)N", "name": "sulfonamide",
             "rationale": "약물유사 골격에서 흔히 쓰이는 안정적 대체기로, 수소결합 donor/acceptor 특성을 일부 유지"},
            {"smiles": "C#N", "name": "nitrile",
             "rationale": "대사 안정성이 개선된 사례가 문헌에 다수 보고됨, 다만 극성은 다소 감소"},
        ],
    },
    "aldehyde": {
        "problem_smarts": "[CX3H1](=O)",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "알데히드의 친전자성(단백질 부가물 형성 우려)을 제거하면서 유사한 형태 유지"},
            {"smiles": "C(O)", "name": "alcohol",
             "rationale": "가장 단순한 환원형 대체, 반응성 크게 감소"},
        ],
    },
    "Michael_acceptor_1": {
        "edit_method": "atom_edit",
        "problem_smarts": "C=CC(=O)",
        "target_idx_pair_in_pattern": (0, 1),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "saturated (C-C single bond)",
             "rationale": "알파,베타-불포화 카르보닐의 C=C 이중결합을 환원하여 "
                          "단백질 친전자성 부가반응(Michael addition, covalent "
                          "binding) 위험을 제거함"},
        ],
    },
    "acid_halide": {
        "problem_smarts": "C(=O)[F,Cl,Br,I]",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "고반응성 아실할라이드를 안정적인 아마이드로 대체"},
            {"smiles": "C(=O)O", "name": "ester",
             "rationale": "아마이드보다 극성이 낮고 유연한 대체 옵션, 가수분해 속도 조절 가능 (검증 필요)"},
        ],
    },
    "alkyl_halide": {
        "problem_smarts": "[Cl,Br,I]",
        "candidates": [
            {"smiles": "O", "name": "hydroxyl (alcohol)",
             "rationale": "이탈기를 제거해 알킬화 반응성을 없앰, 극성은 유사하게 유지"},
            {"smiles": "F", "name": "fluorine",
             "rationale": "할로겐을 유지하되 C-F 결합은 강해 이탈기로 작용하지 않음, 입체적 크기도 유사"},
        ],
    },
    "aniline": {
        "problem_smarts": "[NH2]",
        "candidates": [
            {"smiles": "C(=O)N", "name": "acetamide (acylated amine)",
             "rationale": "1차 방향족 아민을 아마이드로 아실화하여 N-hydroxylation 경로 자체를 차단"},
            {"smiles": "F", "name": "fluorine",
             "rationale": "반응성 아민을 제거하면서 전자끄는기로 고리 전자밀도 보정"},
        ],
    },
    "Sulfonic_acid_2": {
        "problem_smarts": "S(=O)(=O)[OX2H1,OX1-]",
        "candidates": [
            {"smiles": "S(=O)(=O)N", "name": "sulfonamide",
             "rationale": "생리적 pH에서 이온화 정도(전하)를 크게 낮춰 세포막 투과성을 "
                          "개선함. 설폰산은 대부분 음이온 상태로 존재해 경구 흡수가 "
                          "저해되는 경우가 많으나, 설폰아마이드는 유사한 골격을 유지하면서도 "
                          "중성에 가까워 약물유사성이 개선됨"},
            {"smiles": "C(=O)O", "name": "carboxylic acid",
             "rationale": "설폰산보다 산성도가 약하고 부피가 작은 산성 bioisostere "
                          "(검증 필요)"},
        ],
    },
    "imine_1_oxime": {
        "edit_method": "atom_edit",
        "problem_smarts": "C=N[OX2H1]",
        "target_idx_pair_in_pattern": (0, 1),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "amine (reduced)",
             "rationale": "옥심의 C=N 결합을 환원하여, 가수분해 시 원래의 반응성 "
                          "카르보닐(알데히드/케톤)로 되돌아갈 수 있는 대사 불안정 "
                          "경로를 제거함"},
        ],
    },
    "imine_1_general": {
        "edit_method": "atom_edit",
        "problem_smarts": "C=N",
        "target_idx_pair_in_pattern": (0, 1),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "amine (reduced)",
             "rationale": "일반 이민(C=N-R)을 환원하여 가수분해 시 반응성 카르보닐로 "
                          "되돌아갈 수 있는 대사 불안정 경로를 제거함. 옥심 특유의 "
                          "메커니즘보다는 근거가 다소 약하며, 하위 구조별 개별 검증 필요"},
        ],
    },
    "catechol": {
        "edit_method": "atom_edit",
        "problem_smarts": "[OX2H;$(Oc1ccccc1O)]",
        "target_idx_in_pattern": 0,
        "candidates": [
            {"edit_type": "add_substituent", "param": "C", "name": "methoxy",
             "rationale": "인체의 COMT(catechol-O-methyltransferase) 효소가 카테콜을 "
                          "메톡시페놀로 메틸화하여 해독하는 생리적 경로와 동일한 원리. "
                          "오르토-퀴논으로의 산화 경로를 차단하여 세포독성/유전독성 우려를 "
                          "낮춤 (학생 확인 예정: ScienceDirect catechol overview, "
                          "PMC6643002 등 참고)"},
        ],
    },
    "Thiocarbonyl_group": {
        "edit_method": "atom_edit",
        "problem_smarts": "[#6]=[#16]",
        "target_idx_in_pattern": 1,
        "candidates": [
            {"edit_type": "replace_element", "param": 8, "name": "carbonyl (O replacing S)",
             "rationale": "황을 산소로 대체(티오카르보닐->카르보닐)하는 것은 흔한 "
                          "bioisostere 전략으로, 갑상선 기능 저해 등 황 함유 작용기 "
                          "특유의 대사/독성 우려를 낮춤 (검증 필요, thiourea->urea "
                          "치환 논리와 동일 계열)"},
        ],
    },
}

def get_replacement_candidates(rule_name: str) -> dict | None:
    """rule_name에 해당하는 치환 정보(SMARTS + 후보 리스트)를 반환. 없으면 None."""
    return REPLACEMENT_LIBRARY.get(rule_name)

Overwriting src/tools/replacement_library.py


In [ ]:
importlib.reload(src.tools.replacement_library)
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import propose_fix

# 아까 실패했던 케이스들 재확인
test_cases_michael = [
    "CC(=O)/C=C/C=C1/C2CCC(C2)C1(C)C",
    "C=CCOC(=O)C(=C)C",
    "COc1ccc(C(=O)/C(Br)=C\\C(=O)[O-])cc1",
]
for smi in test_cases_michael:
    print(propose_fix(smi, "Michael_acceptor_1", candidate_idx=0))

{'new_smiles': 'CC(=O)CC/C=C1/C2CCC(C2)C1(C)C', 'candidate_used': 'saturated (C-C single bond)', 'rationale': '알파,베타-불포화 카르보닐의 C=C 이중결합을 환원하여 단백질 친전자성 부가반응(Michael addition, covalent binding) 위험을 제거함', 'is_valid': True}
{'new_smiles': 'C=CCOC(=O)C(C)C', 'candidate_used': 'saturated (C-C single bond)', 'rationale': '알파,베타-불포화 카르보닐의 C=C 이중결합을 환원하여 단백질 친전자성 부가반응(Michael addition, covalent binding) 위험을 제거함', 'is_valid': True}
{'new_smiles': 'COc1ccc(C(=O)C(Br)CC(=O)[O-])cc1', 'candidate_used': 'saturated (C-C single bond)', 'rationale': '알파,베타-불포화 카르보닐의 C=C 이중결합을 환원하여 단백질 친전자성 부가반응(Michael addition, covalent binding) 위험을 제거함', 'is_valid': True}


In [ ]:
test_fail = "COc1ccc(C(=O)/C(Br)=C\\C(=O)[O-])cc1"
mol_check = Chem.MolFromSmiles(test_fail)
pattern = Chem.MolFromSmarts("C=CC(=O)")
matches = mol_check.GetSubstructMatches(pattern)
print("매치들:", matches)

for match in matches:
    for idx in match:
        atom = mol_check.GetAtomWithIdx(idx)
        print(f"  인덱스 {idx}: {atom.GetSymbol()}, 이웃: {[n.GetSymbol() for n in atom.GetNeighbors()]}")

매치들: ((8, 10, 11, 12), (10, 8, 6, 7))
  인덱스 8: C, 이웃: ['C', 'Br', 'C']
  인덱스 10: C, 이웃: ['C', 'C']
  인덱스 11: C, 이웃: ['C', 'O', 'O']
  인덱스 12: O, 이웃: ['C']
  인덱스 10: C, 이웃: ['C', 'C']
  인덱스 8: C, 이웃: ['C', 'Br', 'C']
  인덱스 6: C, 이웃: ['C', 'O', 'C']
  인덱스 7: O, 이웃: ['C']


In [ ]:
mol_check2 = Chem.MolFromSmiles(test_fail)
for bond in mol_check2.GetBonds():
    if bond.GetBondTypeAsDouble() == 2.0 and bond.GetBeginAtom().GetSymbol() == 'C' and bond.GetEndAtom().GetSymbol() == 'C':
        print(f"C=C 이중결합: {bond.GetBeginAtomIdx()} - {bond.GetEndAtomIdx()}")

C=C 이중결합: 8 - 10


In [ ]:
result_full = propose_fix(test_fail, "Michael_acceptor_1", candidate_idx=0)
print(result_full)

check_mol = Chem.MolFromSmiles(result_full['new_smiles'])
for atom in check_mol.GetAtoms():
    print(f"{atom.GetSymbol()} idx={atom.GetIdx()}: TotalNumHs={atom.GetTotalNumHs()}, Degree={atom.GetDegree()}, NoImplicit={atom.GetNoImplicit()}, FormalCharge={atom.GetFormalCharge()}")

{'new_smiles': 'COc1ccc(C(=O)C(Br)CC(=O)[O-])cc1', 'candidate_used': 'saturated (C-C single bond)', 'rationale': '알파,베타-불포화 카르보닐의 C=C 이중결합을 환원하여 단백질 친전자성 부가반응(Michael addition, covalent binding) 위험을 제거함', 'is_valid': True}
C idx=0: TotalNumHs=3, Degree=1, NoImplicit=False, FormalCharge=0
O idx=1: TotalNumHs=0, Degree=2, NoImplicit=False, FormalCharge=0
C idx=2: TotalNumHs=0, Degree=3, NoImplicit=False, FormalCharge=0
C idx=3: TotalNumHs=1, Degree=2, NoImplicit=False, FormalCharge=0
C idx=4: TotalNumHs=1, Degree=2, NoImplicit=False, FormalCharge=0
C idx=5: TotalNumHs=0, Degree=3, NoImplicit=False, FormalCharge=0
C idx=6: TotalNumHs=0, Degree=3, NoImplicit=False, FormalCharge=0
O idx=7: TotalNumHs=0, Degree=1, NoImplicit=False, FormalCharge=0
C idx=8: TotalNumHs=1, Degree=3, NoImplicit=False, FormalCharge=0
Br idx=9: TotalNumHs=0, Degree=1, NoImplicit=False, FormalCharge=0
C idx=10: TotalNumHs=2, Degree=2, NoImplicit=False, FormalCharge=0
C idx=11: TotalNumHs=0, Degree=3, NoImplicit=Fals

In [ ]:
%%writefile src/tools/atom_editor.py
from rdkit import Chem


def apply_atom_edit_from_rule(smiles: str, rule_name: str, candidate_idx: int = 0):
    """replacement_library의 atom_edit 규칙을 이용해 원자/결합 직접 편집을 수행."""
    from src.tools.replacement_library import get_replacement_candidates
    info = get_replacement_candidates(rule_name)
    if info is None or info.get("edit_method") != "atom_edit":
        return None
    if candidate_idx >= len(info["candidates"]):
        return None

    candidate = info["candidates"][candidate_idx]
    smarts = info["problem_smarts"]

    mol = Chem.MolFromSmiles(smiles)
    pattern = Chem.MolFromSmarts(smarts)
    if mol is None or pattern is None:
        return None

    matches = mol.GetSubstructMatches(pattern)
    if not matches:
        return None
    match = matches[0]

    rwmol = Chem.RWMol(mol)
    edit_type = candidate["edit_type"]

    if edit_type == "replace_element":
        target_idx = match[info["target_idx_in_pattern"]]
        atom = rwmol.GetAtomWithIdx(target_idx)
        atom.SetAtomicNum(candidate["param"])

    elif edit_type == "add_substituent":
        target_idx = match[info["target_idx_in_pattern"]]
        frag = Chem.MolFromSmiles(candidate["param"])
        if frag is None:
            return None
        combined = Chem.CombineMols(rwmol.GetMol(), frag)
        rwmol = Chem.RWMol(combined)
        offset = mol.GetNumAtoms()
        rwmol.AddBond(target_idx, offset, Chem.BondType.SINGLE)
        atom = rwmol.GetAtomWithIdx(target_idx)
        if atom.GetNumExplicitHs() > 0:
            atom.SetNumExplicitHs(atom.GetNumExplicitHs() - 1)
        else:
            atom.SetNoImplicit(False)

    elif edit_type == "reduce_bond":
        idx1 = match[info["target_idx_pair_in_pattern"][0]]
        idx2 = match[info["target_idx_pair_in_pattern"][1]]
        bond = rwmol.GetBondBetweenAtoms(idx1, idx2)
        if bond is None:
            return None
        bond.SetBondType(Chem.BondType.SINGLE)
        for idx in (idx1, idx2):
            atom = rwmol.GetAtomWithIdx(idx)
            atom.SetNoImplicit(False)
    else:
        return None

    try:
        new_mol = rwmol.GetMol()
        Chem.SanitizeMol(new_mol)
    except Exception:
        return None

    new_smiles = Chem.MolToSmiles(new_mol)

    # 유효성 검증: 파싱 가능 여부 + 전하를 고려한 비정상 원자가 체크
    # (전하가 있는 원자는 결합수가 적어도 정상일 수 있으므로 FormalCharge==0인 경우만 검사)
    check_mol = Chem.MolFromSmiles(new_smiles)
    is_valid = check_mol is not None
    if is_valid:
        for atom in check_mol.GetAtoms():
            if (atom.GetNoImplicit() and atom.GetFormalCharge() == 0
                    and atom.GetSymbol() in ('C', 'N', 'O')
                    and atom.GetTotalNumHs() == 0 and atom.GetDegree() < 4):
                is_valid = False
                break

    return {
        "new_smiles": new_smiles,
        "candidate_used": candidate["name"],
        "rationale": candidate["rationale"],
        "is_valid": is_valid,
    }

Overwriting src/tools/atom_editor.py


In [ ]:
importlib.reload(src.tools.atom_editor)
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import propose_fix

result_recheck = propose_fix(test_fail, "Michael_acceptor_1", candidate_idx=0)
print(result_recheck)

{'new_smiles': 'COc1ccc(C(=O)C(Br)CC(=O)[O-])cc1', 'candidate_used': 'saturated (C-C single bond)', 'rationale': '알파,베타-불포화 카르보닐의 C=C 이중결합을 환원하여 단백질 친전자성 부가반응(Michael addition, covalent binding) 위험을 제거함', 'is_valid': True}


In [ ]:
!git add src/tools/replacement_library.py src/tools/atom_editor.py
!git status

On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	laidd-2026/

nothing added to commit but untracked files present (use "git add" to track)


In [ ]:
!git commit -m "Convert Michael_acceptor_1 to atom_edit (reduce_bond) after discovering fragment-cut approach fails on conjugated/ring-embedded C=C systems (most common failure mode in valid set stuck cases); fix is_valid check to exclude charged atoms (e.g. carboxylate O-) from the abnormal-valence heuristic"
!git push https://{token}@github.com/Dec32th/laidd-2026.git

On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	laidd-2026/

nothing added to commit but untracked files present (use "git add" to track)
Everything up-to-date


In [ ]:
print(propose_fix("O=C(O)CCl", "alkyl_halide", candidate_idx=0))
print(propose_fix("N#CC(C#N)=Cc1ccc(O)c(O)c1", "catechol", candidate_idx=0))
print(propose_fix("CC(C)N1C(=O)N(c2ccccc2)CSC1=NC(C)(C)C", "imine_1_general", candidate_idx=0))

{'new_smiles': 'O=C(O)CO', 'candidate_used': 'hydroxyl (alcohol)', 'rationale': '이탈기를 제거해 알킬화 반응성을 없앰, 극성은 유사하게 유지', 'is_valid': True}
{'new_smiles': 'COc1ccc(C=C(C#N)C#N)cc1O', 'candidate_used': 'methoxy', 'rationale': '인체의 COMT(catechol-O-methyltransferase) 효소가 카테콜을 메톡시페놀로 메틸화하여 해독하는 생리적 경로와 동일한 원리. 오르토-퀴논으로의 산화 경로를 차단하여 세포독성/유전독성 우려를 낮춤 (학생 확인 예정: ScienceDirect catechol overview, PMC6643002 등 참고)', 'is_valid': True}
{'new_smiles': 'CC(C)N1C(=O)N(c2ccccc2)CSC1NC(C)(C)C', 'candidate_used': 'amine (reduced)', 'rationale': '일반 이민(C=N-R)을 환원하여 가수분해 시 반응성 카르보닐로 되돌아갈 수 있는 대사 불안정 경로를 제거함. 옥심 특유의 메커니즘보다는 근거가 다소 약하며, 하위 구조별 개별 검증 필요', 'is_valid': True}


In [ ]:
rule_based_results_valid_v2 = []
for smi in sample_single_valid:
    result = iterative_fix_loop(smi, max_iterations=10)
    rule_based_results_valid_v2.append({"smiles": smi, "status": result['status'], "steps": len(result['history'])-1})

status_counts_valid_v2 = Counter(r['status'] for r in rule_based_results_valid_v2)
print("규칙기반 결과 (valid set 50개, Michael_acceptor_1 수정 후):")
for status, count in status_counts_valid_v2.items():
    print(f"  {status}: {count}개 ({count/len(rule_based_results_valid_v2)*100:.1f}%)")

total_v2 = len(rule_based_results_valid_v2)
success_v2 = sum(1 for r in rule_based_results_valid_v2 if r['status'] == 'success')
partial_v2 = sum(1 for r in rule_based_results_valid_v2 if r['status'] == 'no_known_fix' and r['steps'] >= 1)
print(f"\n완전 해결: {success_v2}개 ({success_v2/total_v2*100:.1f}%)")
print(f"부분 진전: {partial_v2}개 ({partial_v2/total_v2*100:.1f}%)")
print(f"최소 1단계 이상 개선: {(success_v2+partial_v2)/total_v2*100:.1f}%")

[05:04:25] Incomplete atom labelling, cannot make bond


규칙기반 결과 (valid set 50개, Michael_acceptor_1 수정 후):
  success: 24개 (48.0%)
  no_known_fix: 24개 (48.0%)
  stuck: 2개 (4.0%)

완전 해결: 24개 (48.0%)
부분 진전: 24개 (48.0%)
최소 1단계 이상 개선: 96.0%


[05:04:26] Incomplete atom labelling, cannot make bond


In [ ]:
stuck_final = [r for r in rule_based_results_valid_v2 if r['status'] == 'stuck']
for r in stuck_final:
    detail = iterative_fix_loop(r['smiles'], max_iterations=10)
    print(f"분자: {r['smiles'][:60]}")
    print(f"  실패 이유: {detail.get('reason')}")
    print()

분자: CCOC(=O)/C=C(\C)O[Ti](O/C(C)=C/C(=O)OCC)(OC(C)C)OC(C)C
  실패 이유: 'Michael_acceptor_1' 치환 실패

분자: CCCCN=C=O
  실패 이유: 'aldehyde' 치환 실패



In [ ]:
!git config --global user.email "hyekyeong.w@gmail.com"
!git log --oneline -20

ada6f60 (HEAD -> main) Convert Michael_acceptor_1 to atom_edit (reduce_bond) after discovering fragment-cut approach fails on conjugated/ring-embedded C=C systems (most common failure mode in valid set stuck cases); fix is_valid check to exclude charged atoms (e.g. carboxylate O-) from the abnormal-valence heuristic
fd12991 (origin/main, origin/HEAD) Fix critical bug in reduce_bond edit: SetNoImplicit(True) was blocking automatic H recalculation, producing invalid atoms like [C]/[N]; also strengthen is_valid check to catch this class of error. Add imine_1 subclassification (oxime vs general) via toxicophore_detector post-processing, both now use atom_edit with new reduce_bond type. Library now has 11 rules, all verified.
fba049e Library editing & add conditional logic for catechol
4be53ca Add atom-level edit path (RWMol-based) for rules that can't be handled by fragment-cut approach: catechol (context-dependent recursive SMARTS, add_substituent) and Thiocarbonyl_group (ring-embedded at

In [ ]:
!git push https://{token}@github.com/Dec32th/laidd-2026.git

Everything up-to-date


In [ ]:
!git push https://{token}@github.com/Dec32th/laidd-2026.git

print("\n=== 최근 커밋 확인 ===")
!git log --oneline -5

print("\n=== origin/main과 로컬 HEAD 일치 여부 ===")
!git fetch origin
!git status

Everything up-to-date

=== 최근 커밋 확인 ===
ada6f60 (HEAD -> main) Convert Michael_acceptor_1 to atom_edit (reduce_bond) after discovering fragment-cut approach fails on conjugated/ring-embedded C=C systems (most common failure mode in valid set stuck cases); fix is_valid check to exclude charged atoms (e.g. carboxylate O-) from the abnormal-valence heuristic
fd12991 (origin/main, origin/HEAD) Fix critical bug in reduce_bond edit: SetNoImplicit(True) was blocking automatic H recalculation, producing invalid atoms like [C]/[N]; also strengthen is_valid check to catch this class of error. Add imine_1 subclassification (oxime vs general) via toxicophore_detector post-processing, both now use atom_edit with new reduce_bond type. Library now has 11 rules, all verified.
fba049e Library editing & add conditional logic for catechol
4be53ca Add atom-level edit path (RWMol-based) for rules that can't be handled by fragment-cut approach: catechol (context-dependent recursive SMARTS, add_substituent) 

In [ ]:
!ls laidd-2026/ 2>&1 | head -5

data
docs
models
notebooks
outputs


In [ ]:
!rm -rf laidd-2026
!git status

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
